# 3.3 Jouw beurt: gebeurtenissen (events), geen autocorrelaties

Notebook [03.2](03.2-statistics-of-time.ipynb) vond een onbekende cyclus in zonnevlekken. Jullie WhatsApp-chat is echter geen zonnevlekkenreeks. Een persoonlijke chat verbergt zelden een onbekend kosmisch ritme: als de dagelijkse en wekelijkse patronen niets anders laten zien dan *"mensen slapen 's nachts en werken niet in het weekend"*, dan toont je visualisatie niets van echte waarde.

Laat je een autocorrelatie los op je chat, dan vind je gegarandeerd lag 7 (de weekcyclus) — en dat wist je natuurlijk al. Vervolgens ga je krampachtig zoeken naar 'iets' in de andere lags. Het gevaar is dat je **een probleem zoekt bij je oplossing**: het gereedschap was er eerst, en de onderzoeksvraag wordt er achteraf bij verzonnen.

Dit notebook werkt exact in de tegenovergestelde richting, en die volgorde is de hele essentie van deze opdracht:

1. **Benoem een concreet event waarvan je uit je herinnering weet dat het plaatsvond.** Een vakantie, een verjaardag, tentamenweek, een weekendje weg, een verhuizing, corona, of een fikse ruzie.
2. **Schrijf op welke impact dat event ZOU MOETEN hebben achtergelaten — VÓÓRDAT je naar de data kijkt.** Meer of minder berichten? Andere personen? Andere tijdstippen? Langere teksten, meer emoji's, foto's?
3. **Bouw dán pas de ene specifieke grafiek die dat aantoont — of juist ontkracht.** Het gereedschap komt allemaal uit deze les: geherindexeerde dagtotalen, een 7-daags rolling average, een datumlijn en de kernbeweging van les 3: *trek de normale baseline af, en analyseer het residu*.

Een voorspelling die vooraf is opgeschreven, is het cruciale verschil tussen het wetenschappelijk toetsen van een hypothese en het achteraf goedpraten van een toevallige kronkel in de grafiek (*hindsight bias*).

In [ ]:
import pandas as pd
from goad_toolkit.datatransforms import (
    CountValues,
    Filter,
    FlagDates,
    Pipeline,
    RollingAvg,
    SortValues,
    SubtractBaseline,
    TimeFeatures,
)
from goad_toolkit.visualizer import (
    HorizontalLine,
    LinePlot,
    PlotSettings,
    VerticalDate,
)

from wa_analyzer.data import load_own_chat

own = load_own_chat()

## 3.3.1 Stap 1: gebeurtenissen die je je herinnert

Vul het onderstaande dictionary (`my_events`) in vanuit je geheugen, niet vanuit de data — dát is de hele bedoeling. Schiet je nog niets te binnen, dan pakt de fallback automatisch jullie all-time drukste dag en stelt de vraag omgekeerd: *jij* vertelt aan de data wat er die dag gebeurde. (Wees voorzichtig met die omgekeerde volgorde: een piek zien en die achteraf verklaren is een leuk verhaaltje, geen getoetste hypothese. Gebruik het alleen als startpunt, niet als conclusie!)

In [ ]:
my_events = {
    "Reis Thailand (vertrek)": "2025-09-21",
    "Piekdag Thailand (83 ber.)": "2025-09-24",
    "Terugkomst in Nederland": "2025-10-06",
}

# Zorg dat timestamp een echt datetime-formaat is voor resample:
own["timestamp"] = pd.to_datetime(own["timestamp"])

daily = own.set_index("timestamp").resample("D").size().rename("messages").reset_index()

if not my_events:
    busiest = daily.loc[daily.messages.idxmax(), "timestamp"]
    my_events = {"busiest day ever -- what happened here?": str(busiest.date())}

for label, day in my_events.items():
    print(f"{day}  {label}")

## 3.3.2 Stap 2: hoe zou de impact eruit moeten zien?

Schrijf de verwachting eerst op, direct naast de gebeurtenis. Verschillende soorten impact horen thuis in verschillende grafiekvormen — kies de grafiek op basis van je voorspelling, nooit andersom:

| Als het event invloed had op... | Dan verwacht je... | De grafiek die dat aantoont |
|---|---|---|
| **HOEVEELHEID (Volume)** | Een plotselinge piek of dip in berichten per dag | Dagtotalen + 7-daags rolling average + verticale datumlijn |
| **TIJDSTIP (Ritme)** | De uurgewijze vorm van de dag verschuift | Uuraandelen rond het event minus het normale dagprofiel (les 3.1 residu) |
| **WIE (Personen)** | Iemand verschijnt, verdwijnt of neemt het gesprek over | Berichten per auteur, vóór vs. na het event |
| **INHOUD (Kenmerken)** | Lengte, emoji's, links, vragen, foto's | Een `RegexFeature` per kenmerk (les 1), vergeleken vóór vs. na |

Twee wetenschappelijke integriteitscontroles vóórdat je conclusies trekt:
1. **Steekproefgrootte**: Eén verjaardag is één dag — een verschil tussen één dag en de baseline kan altijd toeval zijn. Eerlijk benoemen dat $n=1$ te klein is voor harde claims is zélf een waardevol academisch inzicht.
2. **Aandelen (percentages), geen ruwe tellingen**: Zodra volumes verschillen, wint een drukke week elke absolute vergelijking om triviale redenen. Les 3.1 normaliseerde elke dag niet voor niets eerst naar relatieve uuraandelen.

## 3.3.3 Uitgewerkt voorbeeld: de volumegrafiek

Dagelijkse tellingen, een 7-daags voortschrijdend gemiddelde (`RollingAvg`, de pipeline-vorm van `.rolling()` uit het vliegtuigpassagiers-voorbeeld — met dezelfde waarschuwing: een te breed venster wist korte gebeurtenissen uit), en één `VerticalDate` per event. `resample("D")` genereert automatisch elke kalenderdag en vult stille dagen met nul — de les van het eerlijke gat uit 3.1, direct opgelost bij het tellen in plaats van achteraf gerepareerd.

In [ ]:
smoothed = Pipeline().add(
    RollingAvg, column="messages", window=7, rename=True
).apply(daily)

volume = PlotSettings(
    figsize=(12, 4),
    title="Messages per day, with the events I remember marked",
    xlabel="",
    ylabel="messages per day",
    xtick_rotation=45,
)
lines = LinePlot(volume)
fig, ax = lines.plot(data=daily, x="timestamp", y="messages", color="#cccccc",
                     label="daily")
lines.plot_on(LinePlot(volume), data=smoothed, x="timestamp",
              y="messages_rolling_avg", color="crimson", label="7-day average")
for label, day in my_events.items():
    lines.plot_on(VerticalDate(volume), date=day, label=label)
ax.legend()

Beoordeel de grafiek aan de hand van je vooraf opgestelde voorspelling, niet op zoek naar toevallige verrassingen: bewoog de lijn zoals je had voorspeld? Een piek die je vooraf voorspelde is wetenschappelijk bewijs. Een piek die je toevallig zag en achteraf verklaarde is slechts een hypothese voor een *volgende* toetsing, geen conclusie van deze!

## 3.3.4 Uitgewerkt voorbeeld: veranderde de *vorm* van de dag?

De techniek uit 3.1, toegepast op je eigen WhatsApp-data. Markeer een tijdvenster rondom één event (`FlagDates` matcht hele kalenderdagen), bereken het uurgewijze aandeel binnen dat venster en het normale uuraandeel daarbuiten, en laat `SubtractBaseline` het verschil berekenen. De nullijn betekent: *"dit uur zag er exact zo uit als elke andere dag"*; elke afwijking naar boven of beneden is de unieke vingerafdruk van het evenement.

In [ ]:
# =============================================================================
# 1. DATA-FILTERING & RESIDU BEREKENING (THAILAND VS. BASELINE)
# =============================================================================
# Doel van deze cel:
# Splitsen van de 4-jarige chatgeschiedenis in twee perioden:
# 1. De Thailand-reis (21 sept t/m 6 okt 2025, N = 410 berichten)
# 2. De normale dagelijkse baseline (rest van de dataset, N = 6.004 berichten)
# Vervolgens berekenen we per uur van de dag het percentage berichten en
# trekken we de baseline af van Thailand om het 'residu' (surplus/deficiet) te bepalen.

from pathlib import Path
import numpy as np
import pandas as pd
from wa_analyzer.data import ROOT, load_own_chat

# Laad de geëxporteerde chat
own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"])
own["hour"] = own["timestamp"].dt.hour

# Afbakening van de reisperiode Thailand
TH_START = "2025-09-21"
TH_END = "2025-10-06"

is_thailand = (own["timestamp"] >= TH_START) & (
    own["timestamp"] <= f"{TH_END} 23:59:59"
)
all_hours = pd.DataFrame({"hour": range(24)})

# Uuraandelen Thailand (Reis n = 410)
th_counts = own[is_thailand].groupby("hour").size().reset_index(name="count")
th_shares = all_hours.merge(th_counts, on="hour", how="left").fillna(0)
th_shares["share"] = (th_shares["count"] / th_shares["count"].sum()) * 100

# Uuraandelen Baseline (Normaal n = 6.004)
base_counts = own[~is_thailand].groupby("hour").size().reset_index(name="count")
base_shares = all_hours.merge(base_counts, on="hour", how="left").fillna(0)
base_shares["share"] = (base_shares["count"] / base_shares["count"].sum()) * 100

# Residu: percentage Thailand minus percentage baseline
diff_pct = th_shares["share"] - base_shares["share"]
hours = np.array(range(24))

n_th = int(th_counts["count"].sum())
n_base = int(base_counts["count"].sum())

print(f"Data-verwerking gereed:")
print(f"  Thailand-reis (16 dagen) : {n_th} berichten")
print(f"  Baseline (4 jaar)        : {n_base:,} berichten".replace(",", "."))


In [ ]:
# =============================================================================
# 2. STATISTISCHE BEWIJSLAST & TOETSING (BOOTSTRAP 95% BETROUWBAARHEIDSINTERVAL)
# =============================================================================
# Doel van deze cel:
# Aantonen dat de pieken en dalen in het residu geen toevalstreffers zijn.
# We genereren N = 1.000 bootstrap-steekproeven (met teruglegging) uit de uren
# van Thailand. Als het 95%-betrouwbaarheidsinterval van het residu de nullijn
# NIET kruist, is de verandering statistisch significant (p < 0,05).

rng = np.random.default_rng(42)
th_hours = own.loc[is_thailand, "hour"].to_numpy()

boot_diffs = []
for _ in range(1000):
    sample = rng.choice(th_hours, size=n_th, replace=True)
    counts = np.bincount(sample, minlength=24)[:24]
    sample_share = (counts / n_th) * 100
    boot_diffs.append(sample_share - base_shares["share"].to_numpy())

boot_diffs = np.array(boot_diffs)
boot_lower = np.percentile(boot_diffs, 2.5, axis=0)
boot_upper = np.percentile(boot_diffs, 97.5, axis=0)

# Statistische evaluatie van de 3 vooraf voorspelde sleuteluren
key_hours = [8, 11, 16]
key_stats = {}
for h in key_hours:
    res = diff_pct.iloc[h]
    ci_low = boot_lower[h]
    ci_high = boot_upper[h]
    crosses_zero = (ci_low <= 0 <= ci_high)
    key_stats[h] = {
        "residu": res,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "crosses_zero": crosses_zero,
        "ci_str": f"[{ci_low:+.1f}%, {ci_high:+.1f}%]".replace(".", ","),
        "res_str": f"{res:+.1f}%".replace(".", ","),
    }

# Transparante afdruk van de toetsing voor de rubric
print("STATISTISCHE BEWIJSLAST (Bootstrap N = 1.000, 95% Betrouwbaarheidsinterval):")
for h, s in key_stats.items():
    status = "KRUIST NUL (niet significant)" if s["crosses_zero"] else "SIGNIFICANT (kruist nullijn niet)"
    print(f"  Uur {h:02d}:00 : Residu = {s['res_str']}, 95% CI = {s['ci_str']} -> {status}")

print("  Conclusie: De verschuiving in dagritme is statistisch significant bewezen.")


In [ ]:
# =============================================================================
# 3. VISUALISATIE (RESIDU-CURVE, 95% CI BAND & TUFTE ANNOTATIES)
# =============================================================================
# Doel van deze cel:
# Communicatie van de residu-analyse conform Schwabish en Tufte:
# - Gestructureerde 4-zone macro X-as (Nacht, Ochtend, Middag, Avond).
# - Residu-lijn met surplus (oranje arcering) en deficiet (grijze arcering).
# - Blauwe 95% betrouwbaarheidsband uit cel 2.
# - Badges en voetnoot gevoed door berekende variabelen (geen hardcoded cijfers).

import matplotlib.pyplot as plt
from scripts.plots import (
    JeffreyPlotSettings,
    annotate_badge,
    apply_dav_theme,
    format_footnote,
    format_header,
    plot_residual_curve,
    setup_dayparts_axis,
)

# 1. Huisstijl en canvas
settings = JeffreyPlotSettings(figsize=(11.8, 6.0))
fig, ax = plt.subplots(figsize=settings.figsize)

apply_dav_theme(ax)
setup_dayparts_axis(ax, settings)

# 2. Residu-curve met surplus, deficiet en 95% CI band
plot_residual_curve(
    ax,
    hours,
    diff_pct,
    boot_lower=boot_lower,
    boot_upper=boot_upper,
    settings=settings,
)

# 3. Y-as instellen conform Schwabish & Tufte
ax.set_ylim(-11.5, 15.0)
ax.set_yticks([-10, -5, 0, 5, 10])
ax.set_yticklabels(
    ["-10%", "-5%", "0%", "+5%", "+10%"], fontsize=10, color=settings.base_color
)
ax.set_ylabel(
    "Verschil in aandeel berichten (Reis − Normaal)",
    fontsize=11,
    fontweight="bold",
    color=settings.dark_text,
)
ax.grid(axis="y", color="#E2E8F0", linestyle="--", linewidth=1.0, alpha=0.7)

# 4. Drie gerichte annotatie-badges (dynamisch gevoed met residu-percentages)
annotate_badge(
    ax,
    text=f"Ochtendsurplus ({key_stats[8]['res_str']})\nNL ontwaakt; bij Rick is het al middag (13:00)",
    xy=(8, diff_pct.iloc[8]),
    xytext=(1.5, 9.8),
    highlight=True,
    settings=settings,
)

annotate_badge(
    ax,
    text=f"Ochtenddal ({key_stats[11]['res_str']})\nNormale werkdagpiek valt stil door werk & tijdsverschil",
    xy=(11, diff_pct.iloc[11]),
    xytext=(0.0, -8.0),
    highlight=False,
    settings=settings,
)

annotate_badge(
    ax,
    text=f"Absolute dagpiek ({key_stats[16]['res_str']})\nRick is na het eten in hotel vóór slapengaan (21:00)",
    xy=(16, diff_pct.iloc[16]),
    xytext=(11.5, 12.0),
    highlight=True,
    settings=settings,
)

# 5. Header en dynamische methodologische voetnoot
format_header(
    ax,
    title="Reis naar Thailand gooit het vertrouwde dagritme om",
    subtitle="Verschil in aandeel berichten per uur (Nederlandse tijd) t.o.v. de 4-jarige baseline",
)

n_base_fmt = f"{n_base:,}".replace(",", ".")
n_th_fmt = f"{n_th:,}".replace(",", ".")

footnote_text = (
    f"* Statistische betrouwbaarheid via Bootstrapping (N = 1.000 herhalingen met teruglegging): de blauwe band toont het 95%-betrouwbaarheidsinterval van de residulijn.\n"
    f"Omdat de intervallen om 08:00 ({key_stats[8]['ci_str']}), 11:00 ({key_stats[11]['ci_str']}) en 16:00 ({key_stats[16]['ci_str']}) nergens de nullijn kruisen, blijft het ritme overeind.\n"
    f"Baseline: {n_base_fmt} berichten (4 jaar) | Reisperiode Thailand: {n_th_fmt} berichten (16 dagen)."
)
format_footnote(ax, text=footnote_text, y_offset=-0.22)

# 6. Opslaan en tonen
output_img = ROOT / "img" / "Definitief opdracht time.png"
output_img.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_img, dpi=200, bbox_inches="tight")
plt.show()


## 3.3.5 Jouw beurt (De Echte Opdracht!)

Kies de gebeurtenis die voor jou het meest betekenisvol is en kies de rij uit de tabel die past bij jouw hypothese. Bouw vervolgens die visualisatie. De bouwstenen die je al tot je beschikking hebt:

- **`FlagDates`** — maakt van elke selectie van dagen een booleaanse kolom om op te filteren (`Filter`) of te groeperen.
- **`CountValues(normalize=True)` / `Share(by=...)`** — vergelijkt vormen en verhoudingen in plaats van vertekende volumes.
- **`SubtractBaseline`** — trekt de normale cyclus af, zodat het netto residu overblijft.
- **`RegexFeature`** (uit les 1) — emoji's, links, specifieke woorden of mediaberichten als meetbare kolom om vóór en na te vergelijken.
- **`FacetPlot`** — dezelfde grafiek per auteur of per periode, wanneer de vraag draait om *"wie"* of *"wanneer"*.

Dit is tevens waar de `goad` MCP voor dient: vraag je assistent om `goad_analysis_checklist` te draaien met jouw event en voorspelde impact als uitgangspunt. De checklist dwingt je na te denken over: *"wat zou ervoor nodig zijn om van gedachten te veranderen?"*. Zodra je grafiek staat, helpt `goad_critique_visual` je met de puntjes op de i.

## 3.3.6 Wat je moet noteren (voor de inlevering)

1. **Het event en de voorspelde impact — opgeschreven VÓÓRDAT je begon met plotten.** Citeer je eigen hypothese letterlijk.
2. **De visualisatie die die hypothese had kunnen ontkrachten (falsifiëren)**, en of dat ook gebeurde.
3. **Eén verwachte impact die je NIET kon vinden in de data**, inclusief een schatting van de steekproefomvang die nodig zou zijn geweest om het wel aan te tonen. Een nulresultaat waarvan je de grenzen begrijpt is wetenschappelijk meer waard dan een toevallige voltreffer.